*[cell 0]*

# Sample description

The analysed sample of one stamped LVL_0 handoff (pinned as `LVL0_HANDOFF` in `[cell 1]`), described from the dataset
the handoff was built from: the file in `PROJECT_ROOT/dset` that the handoff's LVL_0 run log names, checked by its
SHA-256. The intake filters (`max_age`, sessions, conditions) and the age-group split (`age_group_split`) are read from
that run log; the notebook stops unless the filtered dataset has exactly the handoff's participants and sessions. Run it
after the LVL0 notebook; set `PROJECT_ROOT` (README, "Paths") and the pin in `[cell 1]`.

**Optional records** (`[cell 2]`, empty until filled): a ledger of what the dataset does not carry (why a participant
has fewer sessions than the others, participants without data, corrections applied to the dataset before the
pipeline), and `COVARIATES`, participant-level columns of the dataset (e.g. questionnaire scores) to describe in the
sample-characteristics table. Age appears only with an `age` column, the age groups (younger: age < `age_group_split`;
older) only with the split set, and the sex rows only with a `sex` column.

| Export (`LVL_OUTS/<handoff stem>_demographics_<key>.csv`) | Content |
|---|---|
| `ledger` | the optional records: `record, id, category, description`; empty without records |
| `flow` | dataset → age exclusion (with `max_age` set) → analysed sample: `Stage, Participants, Sessions, [Younger, Older,] Note` |
| `sessions` | sessions per participant: `Sessions` (count), `Participants`, one column per condition |
| `fewer_sessions` | participants with fewer analysed sessions than the most any participant has, with the ledger's reason: `id, condition, sessions analysed, category, description` |
| `fewer_sessions_counts` | those participants by reason: `category, n` |
| `balance` | Table 0.1: sample by condition: `Characteristic`, one column per condition, `Total` |
| `table` | sample characteristics by condition, age group and sex: `Group, n, %`, `Age M, Age SD` and the M and SD of each covariate |

In [ ]:
# [cell 1]
# ===== INPUTS: set these before running =====
LVL0_HANDOFF = None   # the LVL_0 handoff whose sample this notebook describes, in LVL_OUTS, e.g. "lv0_out_20261005_101500.csv"
PROJECT_ROOT = "SET_ME"   # the absolute project folder holding code/, dset/ and dervatives/ (README, "Paths"; EMST /home/sjsuss626/EMST_files/sjs_dissertation)

_SET_ME = {"PROJECT_ROOT": (PROJECT_ROOT, 'the absolute project folder holding code/, dset/ and dervatives/; the '
                                          'README\'s one-line sed command sets it (README, "Paths"); EMST '
                                          '/home/sjsuss626/EMST_files/sjs_dissertation')}   # name: (value, what to set)
_unset = [k for k, (v, _) in _SET_ME.items() if isinstance(v, str) and v == "SET_ME"]
if _unset:
    raise SystemExit(f'settings check (demographic_report.ipynb): {len(_unset)} setting(s) still hold "SET_ME". Set each, then run '
                     'again (README, "Settings"):\n' + "\n".join(
                         f"  - {k}  (demographic_report [cell 1])\n      {_SET_ME[k][1]}" for k in _unset))

# =============================================================================
# IMPORTS, INPUT, PATHS
# =============================================================================
import hashlib, json, re, warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=RuntimeWarning)
pd.set_option("display.width", 250)
pd.set_option("display.max_rows", 200)

# ---- locations: everything is inside PROJECT_ROOT ----------------
_OUT = Path(PROJECT_ROOT) / "dervatives" / "practical_application_generic"
LVL_OUTS = _OUT / "lvl_outs"   # the handoff; this notebook's tables

if LVL0_HANDOFF is None:
    raise SystemExit("LVL0_HANDOFF is not set (top of this cell). Name one LVL_0 handoff; "
                     f"available in {LVL_OUTS}: {sorted(p.name for p in LVL_OUTS.glob('lv0_out_*.csv'))}")
LV0_PATH = LVL_OUTS / LVL0_HANDOFF
_m0 = re.match(r"lv0_out_(\d{8}_\d{6})", LV0_PATH.name)
if not _m0 or LV0_PATH.suffix != ".csv" or not LV0_PATH.is_file():
    raise SystemExit(f"LVL0_HANDOFF = {LVL0_HANDOFF!r} is not an existing LVL_0 handoff "
                     "(lv0_out_<YYYYmmdd_HHMMSS>[...].csv)")
_recs = sorted(LV0_PATH.parent.glob(f"lv0_out_{_m0.group(1)}*_parameters.json"))
if len(_recs) != 1:
    raise SystemExit(f"expected exactly one LVL_0 run log lv0_out_{_m0.group(1)}*_parameters.json beside "
                     f"{LV0_PATH.name}, found {[p.name for p in _recs]}")
LVL0_RUN_LOG = json.load(open(_recs[0]))
# The dataset the handoff was built from: the run log's input file, in PROJECT_ROOT/dset.
DSET_PATH = Path(PROJECT_ROOT) / "dset" / Path(LVL0_RUN_LOG["input_long_path"]).name
if not DSET_PATH.is_file():
    raise SystemExit(f"{LV0_PATH.name} was built from {LVL0_RUN_LOG['input_long_path']}; {DSET_PATH} does not exist")
_sha = hashlib.sha256()
with open(DSET_PATH, "rb") as _f:
    for _b in iter(lambda: _f.read(1 << 20), b""):
        _sha.update(_b)
if LVL0_RUN_LOG["outputs"].get("input_long_sha256") != _sha.hexdigest():
    raise SystemExit(f"{DSET_PATH.name} is not the file {LV0_PATH.name} was built from (its run log records "
                     f"{LVL0_RUN_LOG['outputs'].get('input_long_sha256')}); re-run LVL_0 with the current code")
P0 = LVL0_RUN_LOG["parameters"]
for _k in ("max_age", "age_group_split", "sessions", "conditions"):
    if _k not in P0:
        raise SystemExit(f"the LVL_0 run log {_recs[0].name} does not record {_k!r}: re-run LVL_0 with the current code")
AGE_SPLIT = P0["age_group_split"]     # younger = age < AGE_SPLIT, older = the rest; None = no age groups (LVL_0_CONFIG)
OUT_STEM = LVL_OUTS / f"{LV0_PATH.stem}_demographics"
TABLES = {}


def export(key, frame, title):
    """Write one table to LVL_OUTS as <handoff stem>_demographics_<key>.csv and show it."""
    TABLES[key] = frame
    frame.to_csv(f"{OUT_STEM}_{key}.csv", index=False)
    print(title)
    display(frame)


print(f"dataset       : {DSET_PATH}")
print(f"LVL_0 handoff : {LV0_PATH.name}  (run log {_recs[0].name}: max_age {P0['max_age']}, age_group_split "
      f"{AGE_SPLIT}, sessions {P0['sessions']}, conditions {P0['conditions']})")
print(f"outputs       : {OUT_STEM}_*.csv")

# Interpreter and package versions of this run.
import sys as _vsys, platform as _vplatform
from importlib import metadata as _vmeta


def _pkg_version(p):
    try:
        return _vmeta.version(p)
    except _vmeta.PackageNotFoundError:
        return "not installed"


print(f"interpreter: {_vsys.executable} (Python {_vplatform.python_version()})")
print("packages: " + ", ".join(f"{p} {_pkg_version(p)}" for p in (
    "numpy", "pandas", "scipy", "scikit-learn", "statsmodels", "hmmlearn", "ruptures", "joblib", "matplotlib")))

In [ ]:
# [cell 2]
# =============================================================================
# OPTIONAL RECORDS: WHAT THE DATASET DOES NOT CARRY, AND THE COVARIATES TO DESCRIBE
# =============================================================================
# Each may stay empty; the ledger export lists the records (it is empty without any). Ids as in the dataset's id column.
# Why a participant has fewer sessions than the others: {id: (category, description)}, e.g.
# {101: ("incomplete session", "S1 stopped after 184 of 200 stimuli")}; participants without an entry are counted as
# "no ledger entry" (EMST 16 entries).
SESSION_MISSING_REASONS = {}
# Participants with no data in the dataset: {id: description}, e.g. {102: "withdrew before the rating task"}.
WITHDRAWN = {}
# Corrections applied to the dataset before this pipeline (already in it; listed so the sample description can cite
# them): [(field, what, why)], e.g. [("age", "103 -> 25", "age entered at screening, corrected from the records")].
CORRECTIONS = []
# Participant-level columns of the dataset to describe in the sample-characteristics table ([cell 6]): {column: label},
# e.g. {"dep_t": "Depression (T)"} (EMST: dep_t, anx_t); one value per participant (its first row).
COVARIATES = {}
export("ledger", pd.DataFrame(
    [{"record": "missing session", "id": i, "category": c, "description": d} for i, (c, d) in SESSION_MISSING_REASONS.items()]
    + [{"record": "withdrawal", "id": i, "category": "withdrew", "description": d} for i, d in WITHDRAWN.items()]
    + [{"record": f"correction: {k}", "id": np.nan, "category": v, "description": d} for k, v, d in CORRECTIONS],
    columns=["record", "id", "category", "description"]),
    "Ledger: records behind the sample description")

In [ ]:
# [cell 3]
# =============================================================================
# LOAD: THE DATASET, THE INTAKE FILTERS, THE ANALYSED SAMPLE
# =============================================================================
_head = list(pd.read_csv(DSET_PATH, nrows=0).columns)
HAS_AGE, HAS_SEX = "age" in _head, "sex" in _head
_no_col = [c for c in COVARIATES if c not in _head]
if _no_col:
    raise SystemExit(f"COVARIATES ([cell 2]) names {_no_col}, which {DSET_PATH.name} does not have")
_cols = list(dict.fromkeys(["id", "condition", "session"] + (["age"] if HAS_AGE else []) + (["sex"] if HAS_SEX else [])
                           + list(COVARIATES)))
dset = pd.read_csv(DSET_PATH, usecols=_cols, low_memory=False)
# The LVL_0 intake filters (its run log): age (max_age None = none), sessions and conditions ("all" = none).
_keep = pd.Series(True, index=dset.index)
if P0["max_age"] is not None:
    _keep &= dset["age"] <= P0["max_age"]
if P0["sessions"] != "all":
    _keep &= dset["session"].isin(P0["sessions"])
if P0["conditions"] != "all":
    _keep &= dset["condition"].isin(P0["conditions"])
analysed = dset[_keep]
# the analysed sample is the LVL_0 handoff's: same participants, same sessions
_h = pd.read_csv(LV0_PATH, usecols=["id", "session"]).drop_duplicates()
_a = analysed[["id", "session"]].drop_duplicates()
if set(map(tuple, _h.values)) != set(map(tuple, _a.values)):
    raise SystemExit(f"the dataset after the LVL_0 intake filters ({_a.id.nunique()} participants, {len(_a)} sessions) is "
                     f"not the handoff's ({_h.id.nunique()} participants, {len(_h)} sessions)")
CONDS = sorted(analysed["condition"].unique())
_group = lambda age: np.where(age < AGE_SPLIT, "younger", "older")
subj = analysed.drop_duplicates("id")[[c for c in _cols if c != "session"]].reset_index(drop=True)
if AGE_SPLIT is not None:
    subj["group"] = _group(subj["age"])
subj["n_sessions"] = subj["id"].map(_a.groupby("id").size())
print(f"analysed sample: {len(subj)} participants"
      + (f" ({(subj.group == 'younger').sum()} younger than {AGE_SPLIT}, {(subj.group == 'older').sum()} older)"
         if AGE_SPLIT is not None else "")
      + f", {len(_a)} sessions, {len(analysed):,} responses = the LVL_0 handoff")

In [ ]:
# [cell 4]
# =============================================================================
# SAMPLE FLOW, SESSIONS PER PARTICIPANT, PARTICIPANTS WITH FEWER SESSIONS
# =============================================================================
_all = dset.drop_duplicates("id")[[c for c in ("id", "age", "condition") if c in dset.columns]].copy()
if AGE_SPLIT is not None:
    _all["group"] = _group(_all["age"])
_ses_all = dset.drop_duplicates(["id", "session"])
_ages = lambda d: ({"Younger": int((d.group == "younger").sum()), "Older": int((d.group == "older").sum())}
                   if AGE_SPLIT is not None else {})
_flow = [{"Stage": f"dataset ({DSET_PATH.name})", "Participants": len(_all), "Sessions": len(_ses_all), **_ages(_all),
          "Note": ""}]
if P0["max_age"] is not None:
    _old = _all[_all.age > P0["max_age"]]
    _flow.append({"Stage": f"excluded: older than {P0['max_age']}", "Participants": len(_old),
                  "Sessions": int(_ses_all.id.isin(_old.id).sum()), **_ages(_old),
                  "Note": (f"ages {_old.age.min():.0f}-{_old.age.max():.0f}; M = {_old.age.mean():.2f}, "
                           f"SD = {_old.age.std():.2f}") if len(_old) else ""})
_filters = [f"{k} {P0[k]}" for k in ("sessions", "conditions") if P0[k] != "all"]
_flow.append({"Stage": "analysed sample (LVL_0 handoff)", "Participants": len(subj), "Sessions": len(_a), **_ages(subj),
              "Note": ", ".join(f"{c} {int((subj.condition == c).sum())}" for c in CONDS) + " participants"
                      + (f"; intake filters: {', '.join(_filters)}" if _filters else "")})
export("flow", pd.DataFrame(_flow), "Sample flow")

_ses = (subj.groupby(["n_sessions", "condition"]).size().unstack("condition", fill_value=0)
        .reindex(columns=CONDS, fill_value=0).rename_axis(index="Sessions", columns=None))
_ses.insert(0, "Participants", _ses.sum(axis=1))
export("sessions", _ses.reset_index(), "Sessions per participant, by condition")

# Fewer sessions than the most any analysed participant has, with the ledger's reason (SESSION_MISSING_REASONS).
_max_ses = int(subj.n_sessions.max())
_fewer = (subj.loc[subj.n_sessions < _max_ses, ["id", "condition", "n_sessions"]]
          .rename(columns={"n_sessions": "sessions analysed"}).sort_values(["condition", "id"]).reset_index(drop=True))
_fewer["category"] = _fewer.id.map(lambda i: SESSION_MISSING_REASONS.get(i, ("no ledger entry", ""))[0])
_fewer["description"] = _fewer.id.map(lambda i: SESSION_MISSING_REASONS.get(i, ("", ""))[1])
export("fewer_sessions", _fewer, f"Participants with fewer than {_max_ses} analysed sessions, with the ledger's reason")
export("fewer_sessions_counts", _fewer.groupby("category").size().rename("n").reset_index(),
       "Participants with fewer sessions, by recorded reason")

In [ ]:
# [cell 5]
# =============================================================================
# TABLE 0.1: SAMPLE BY CONDITION
# =============================================================================
# Participants, sessions per session number, sex (as recorded) when the dataset has a sex column, the younger group
# when age_group_split is set, and age when the dataset has an age column; one column per condition and the total.
_ss = _a.merge(subj[["id", "condition"]], on="id")
_pct = lambda m: f"{int(m.sum())} ({100 * m.mean():.1f})"
_chars = [("Participants", lambda p, s: len(p))]
_chars += [(f"Sessions, S{k}", lambda p, s, k=k: int((s.session == k).sum())) for k in sorted(_ss.session.unique())]
if HAS_SEX:
    _chars += [(f"Sex: {lv}, n (%)", lambda p, s, lv=lv: _pct(p.sex == lv)) for lv in sorted(subj.sex.dropna().unique())]
if AGE_SPLIT is not None:
    _chars += [(f"Younger (< {AGE_SPLIT}), n (%)", lambda p, s: _pct(p.group == "younger"))]
if HAS_AGE:
    _chars += [("Age, Mdn [Q1, Q3]",
                lambda p, s: f"{p.age.median():.0f} [{p.age.quantile(.25):.0f}, {p.age.quantile(.75):.0f}]")]
rows = []
for lab, fn in _chars:
    r = {"Characteristic": lab}
    for cd in CONDS:
        r[cd] = fn(subj[subj.condition == cd], _ss[_ss.condition == cd])
    r["Total"] = fn(subj, _ss)
    rows.append(r)
export("balance", pd.DataFrame(rows), "Table 0.1. Sample by condition")

In [ ]:
# [cell 6]
# =============================================================================
# SAMPLE CHARACTERISTICS BY CONDITION, AGE GROUP AND SEX
# =============================================================================
# One row per participant. n, % of the analysed sample, and M (SD) of age and of each covariate ([cell 2]), for the
# whole sample, each condition, each age group and each sex level.
N_SAMPLE = len(subj)


def summarize(g, label):
    r = {"Group": label, "n": len(g), "%": 100 * len(g) / N_SAMPLE}
    if HAS_AGE:
        r.update({"Age M": g.age.mean(), "Age SD": g.age.std()})
    for col, lab in COVARIATES.items():
        v = pd.to_numeric(g[col], errors="coerce")
        r.update({f"{lab} M": v.mean(), f"{lab} SD": v.std()})
    return r


rows = [summarize(subj, "Overall")] + [summarize(subj[subj.condition == c], f"Condition {c}") for c in CONDS]
if AGE_SPLIT is not None:
    rows += [summarize(subj[subj.group == "younger"], f"Younger (< {AGE_SPLIT})"),
             summarize(subj[subj.group == "older"], f"Older (≥ {AGE_SPLIT})")]
if HAS_SEX:
    rows += [summarize(subj[subj.sex == s], f"Sex: {s}") for s in sorted(subj.sex.dropna().unique())]
export("table", pd.DataFrame(rows), "Sample characteristics by condition, age group and sex")